Visualizations created during the development of the model herein can be found here: https://www.kaggle.com/db102291/titanic-competition-visualization-w-seaborn

In [1]:
from xgboost import XGBClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GridSearchCV
import category_encoders as ce
from sklearn.pipeline import Pipeline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd 

In [2]:
#Import training and testing data
train_data = pd.read_csv("../input/titanic/train.csv")
test_data = pd.read_csv("../input/titanic/test.csv")

In [3]:
#Which columns have missing values?
display(train_data.isnull().sum().sort_values(ascending=False))
display(test_data.isnull().sum().sort_values(ascending=False))

Cabin          687
Age            177
Embarked         2
Fare             0
Ticket           0
Parch            0
SibSp            0
Sex              0
Name             0
Pclass           0
Survived         0
PassengerId      0
dtype: int64

Cabin          327
Age             86
Fare             1
Embarked         0
Ticket           0
Parch            0
SibSp            0
Sex              0
Name             0
Pclass           0
PassengerId      0
dtype: int64

In [4]:
#Create new Cabin variable
train_data['Cabin_new'] = train_data['Cabin'].str[0]
test_data['Cabin_new'] = train_data['Cabin'].str[0]

#Create title variable
train_data['Title']=train_data.Name.str.extract('([A-Za-z]+)\.')
test_data['Title']=test_data.Name.str.extract('([A-Za-z]+)\.')

#Create Fam_size variable
train_data['Fam_size'] = train_data['SibSp'] + train_data['Parch']
test_data['Fam_size'] = test_data['SibSp'] + test_data['Parch']

In [5]:
train_data

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Cabin_new,Title,Fam_size
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S,NaN,Mr,1
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C,C,Mrs,1
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,NaN,Miss,0
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S,C,Mrs,1
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S,NaN,Mr,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,NaN,S,NaN,Rev,0
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,B42,S,B,Miss,0
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.4500,NaN,S,NaN,Miss,3
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.0000,C148,C,C,Mr,0


In [6]:
#Preprocessing numerical data
#Fare
train_data['Fare'].fillna(train_data['Fare'].median(), inplace=True)
test_data['Fare'].fillna(train_data['Fare'].mean(), inplace=True)

#Age
train_data['Age'].fillna(train_data['Age'].mean(), inplace=True)
test_data['Age'].fillna(train_data['Age'].mean(), inplace=True)

In [7]:
features = ["Pclass", "Age", "Fam_size", "Fare", "Sex", "Embarked"]
cat_cols = ['Sex', 'Embarked', "Pclass"]
num_cols = ['Age', 'Fam_size', 'Fare']

y = train_data["Survived"]
X = train_data[features]
X_test = test_data[features]

In [8]:
#Preprocessing for categorical data
cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

#Bundle preprocessing for numerical and categorical data
preprocessor = ColumnTransformer(transformers=[('cat', cat_transformer, cat_cols)])

In [9]:
#Model
model = XGBClassifier(random_state=126)

#Bundle preprocessing and modeling code in a pipeline
my_pipeline = Pipeline(steps=[('preprocess', preprocessor), 
                              ('model', model)])

param_grid = {
    'model__n_estimators': [20, 40, 60],
    'model__learning_rate': [0.05, 0.07, 0.08],
    'model__max_depth': [4, 6, 8]}


search = GridSearchCV(my_pipeline, param_grid, n_jobs=-1, verbose=10, cv=10)
search.fit(X, y)
print("Best parameter (CV score=%0.3f):" % search.best_score_)
print(search.best_params_)

Fitting 10 folds for each of 27 candidates, totalling 270 fits


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=-1)]: Done   5 tasks      | elapsed:    2.7s
[Parallel(n_jobs=-1)]: Done  10 tasks      | elapsed:    2.8s
[Parallel(n_jobs=-1)]: Done  17 tasks      | elapsed:    2.9s
[Parallel(n_jobs=-1)]: Done  24 tasks      | elapsed:    3.1s
[Parallel(n_jobs=-1)]: Batch computation too fast (0.1966s.) Setting batch_size=2.
[Parallel(n_jobs=-1)]: Done  33 tasks      | elapsed:    3.3s
[Parallel(n_jobs=-1)]: Batch computation too fast (0.1652s.) Setting batch_size=4.
[Parallel(n_jobs=-1)]: Done  44 tasks      | elapsed:    3.6s
[Parallel(n_jobs=-1)]: Done  76 tasks      | elapsed:    4.3s
[Parallel(n_jobs=-1)]: Done 120 tasks      | elapsed:    5.1s
[Parallel(n_jobs=-1)]: Done 172 tasks      | elapsed:    6.1s
[Parallel(n_jobs=-1)]: Done 224 tasks      | elapsed:    7.1s


Best parameter (CV score=0.811):
{'model__learning_rate': 0.05, 'model__max_depth': 4, 'model__n_estimators': 20}


[Parallel(n_jobs=-1)]: Done 263 out of 270 | elapsed:    7.8s remaining:    0.2s
[Parallel(n_jobs=-1)]: Done 270 out of 270 | elapsed:    7.9s finished


In [10]:
predictions = search.predict(X_test)

output = pd.DataFrame({'PassengerId': test_data.PassengerId, 'Survived': predictions})
output.to_csv('my_submission.csv', index=False)
print("Your submission was successfully saved!")

Your submission was successfully saved!
